# Geo-Holdout Experiment Design and Power Analysis

## Measurement 360 Portfolio

This notebook designs a synthetic geo-holdout experiment using the eligible
United States regions identified in the BigQuery analytical warehouse.

### Objectives

1. Load and validate eligible PRE-period geo-week observations.
2. Evaluate regional stability and scale.
3. Match comparable treatment and control regions using PRE-period outcomes.
4. Estimate pre-treatment balance and statistical power.
5. Freeze the experiment design before examining TEST-period outcomes.

### Data boundary

- Observed source: Public GA4 ecommerce data.
- Eligible regions: 20 United States regions.
- PRE period: 8 weeks.
- TEST period: 5 weeks.
- The TEST-period outcomes are not used during matching or treatment assignment.
- Treatment effects will be simulated transparently because the public dataset
  does not contain a real randomized media intervention.

In [7]:
from google.colab import auth
from google.cloud import bigquery

import numpy as np
import pandas as pd

PROJECT_ID = "measurement-360-portfolio"
LOCATION = "US"

auth.authenticate_user(project_id=PROJECT_ID)

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

print("Authenticated project:", client.project)
print("Default BigQuery location:", client.location)

Authenticated project: measurement-360-portfolio
Default BigQuery location: US


In [8]:
PRE_PERIOD_QUERY = """
SELECT
    geo_key,
    geo_country,
    geo_region,
    week_start_date,
    week_end_date,
    analysis_week_number,
    period_week_number,
    observed_session_count,
    observed_unique_users,
    observed_engaged_sessions,
    observed_new_user_sessions,
    observed_transaction_count,
    observed_revenue_usd,
    observed_item_quantity,
    observed_transaction_rate,
    observed_revenue_per_session_usd,
    observed_average_order_value_usd,
    pre_session_count,
    pre_transaction_count,
    pre_revenue_usd,
    pre_revenue_coefficient_of_variation,
    data_origin,
    treatment_assignment_status,
    is_synthetic_outcome
FROM
    `measurement-360-portfolio.measurement_360_mart.fct_geo_weekly_outcomes`
WHERE
    meets_initial_eligibility = TRUE
    AND period_name = 'PRE'
ORDER BY
    geo_region,
    week_start_date
"""

job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=50 * 1024 * 1024,
    use_query_cache=True
)

query_job = client.query(
    PRE_PERIOD_QUERY,
    job_config=job_config,
    location=LOCATION
)

pre_df = query_job.to_dataframe()

processed_mb = (query_job.total_bytes_processed or 0) / (1024 ** 2)
billed_mb = (query_job.total_bytes_billed or 0) / (1024 ** 2)

print("Query status:", query_job.state)
print(f"Bytes processed: {processed_mb:.2f} MB")
print(f"Bytes billed: {billed_mb:.2f} MB")
print("Dataframe shape:", pre_df.shape)

Query status: DONE
Bytes processed: 0.00 MB
Bytes billed: 0.00 MB
Dataframe shape: (160, 24)


In [9]:
pre_df["week_start_date"] = pd.to_datetime(pre_df["week_start_date"])
pre_df["week_end_date"] = pd.to_datetime(pre_df["week_end_date"])

display(pre_df.head())

print("Rows:", len(pre_df))
print("Regions:", pre_df["geo_region"].nunique())
print("PRE weeks:", pre_df["week_start_date"].nunique())
print("First PRE week:", pre_df["week_start_date"].min().date())
print("Last PRE week:", pre_df["week_end_date"].max().date())

print("\nRows per region:")
print(
    pre_df.groupby("geo_region")
    .size()
    .value_counts()
    .sort_index()
)

,geo_key,geo_country,geo_region,week_start_date,week_end_date,analysis_week_number,period_week_number,observed_session_count,observed_unique_users,observed_engaged_sessions,...,observed_transaction_rate,observed_revenue_per_session_usd,observed_average_order_value_usd,pre_session_count,pre_transaction_count,pre_revenue_usd,pre_revenue_coefficient_of_variation,data_origin,treatment_assignment_status,is_synthetic_outcome
0,United States | Arizona,United States,Arizona,2020-11-01,2020-11-07,1,1,196,142,160,...,0.015306,0.698980,45.666667,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
1,United States | Arizona,United States,Arizona,2020-11-08,2020-11-14,2,2,118,90,100,...,0.008475,0.186441,22.000000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
2,United States | Arizona,United States,Arizona,2020-11-15,2020-11-21,3,3,172,125,145,...,0.029070,1.709302,58.800000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
3,United States | Arizona,United States,Arizona,2020-11-22,2020-11-28,4,4,147,123,129,...,0.006803,0.959184,141.000000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
4,United States | Arizona,United States,Arizona,2020-11-29,2020-12-05,5,5,194,152,166,...,0.046392,3.943299,85.000000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False


Rows: 160
Regions: 20
PRE weeks: 8
First PRE week: 2020-11-01
Last PRE week: 2020-12-26

Rows per region:
8    20
Name: count, dtype: int64


In [10]:
required_columns = {
    "geo_key",
    "geo_region",
    "week_start_date",
    "observed_session_count",
    "observed_transaction_count",
    "observed_revenue_usd",
    "pre_revenue_coefficient_of_variation",
    "data_origin",
    "treatment_assignment_status",
    "is_synthetic_outcome"
}

assert required_columns.issubset(pre_df.columns), \
    "One or more required fields are missing."

assert len(pre_df) == 160, \
    "Expected 160 eligible PRE-period region-week rows."

assert pre_df["geo_region"].nunique() == 20, \
    "Expected 20 eligible regions."

assert pre_df["week_start_date"].nunique() == 8, \
    "Expected eight PRE-period weeks."

assert pre_df.groupby("geo_region").size().eq(8).all(), \
    "Every eligible region must contain exactly eight PRE weeks."

assert pre_df["geo_key"].notna().all(), \
    "Every observation must have a geo key."

assert (
    pre_df[
        [
            "observed_session_count",
            "observed_transaction_count",
            "observed_revenue_usd"
        ]
    ] >= 0
).all().all(), "Outcome values cannot be negative."

assert pre_df["data_origin"].eq("PUBLIC_GA4_OBSERVED").all(), \
    "PRE observations must come from observed GA4 data."

assert pre_df["treatment_assignment_status"].eq("UNASSIGNED").all(), \
    "Treatment must remain unassigned during baseline extraction."

assert pre_df["is_synthetic_outcome"].eq(False).all(), \
    "PRE-period observations must not be synthetic."

assert int(pre_df["observed_session_count"].sum()) == 81006, \
    "PRE-period sessions do not reconcile."

assert int(pre_df["observed_transaction_count"].sum()) == 1499, \
    "PRE-period transactions do not reconcile."

assert np.isclose(
    pre_df["observed_revenue_usd"].sum(),
    104458.00,
    atol=0.01
), "PRE-period revenue does not reconcile."

print("PRE-PERIOD EXTRACTION: PASS")

PRE-PERIOD EXTRACTION: PASS


In [11]:
region_summary = (
    pre_df
    .groupby("geo_region", as_index=False)
    .agg(
        pre_weeks=("week_start_date", "nunique"),
        pre_sessions=("observed_session_count", "sum"),
        pre_transactions=("observed_transaction_count", "sum"),
        pre_revenue_usd=("observed_revenue_usd", "sum"),
        average_weekly_revenue_usd=(
            "observed_revenue_usd",
            "mean"
        ),
        weekly_revenue_stddev_usd=(
            "observed_revenue_usd",
            "std"
        ),
        revenue_coefficient_of_variation=(
            "pre_revenue_coefficient_of_variation",
            "first"
        )
    )
    .sort_values(
        ["pre_revenue_usd", "pre_sessions"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(region_summary)

print("Summary rows:", len(region_summary))
print("Summary sessions:", region_summary["pre_sessions"].sum())
print("Summary transactions:", region_summary["pre_transactions"].sum())
print(
    "Summary revenue:",
    round(region_summary["pre_revenue_usd"].sum(), 2)
)

,geo_region,pre_weeks,pre_sessions,pre_transactions,pre_revenue_usd,average_weekly_revenue_usd,weekly_revenue_stddev_usd,revenue_coefficient_of_variation
0,California,8,20964,383,27274.0,3409.250,1630.179285,0.478164
1,Texas,8,7412,165,11680.0,1460.000,775.586230,0.531223
2,Virginia,8,6498,117,7873.0,984.125,797.514610,0.810379
3,Florida,8,4575,86,7084.0,885.500,653.989952,0.738554
4,New York,8,6717,108,5468.0,683.500,405.389055,0.593108
5,New Jersey,8,2948,46,5224.0,653.000,493.880842,0.756326
6,Massachusetts,8,3929,68,4742.0,592.750,326.129575,0.550198
7,North Carolina,8,2471,46,3905.0,488.125,438.166452,0.897652
8,Illinois,8,3757,62,3757.0,469.625,346.690450,0.738228
9,Michigan,8,2464,46,3678.0,459.750,334.639998,0.727874


Summary rows: 20
Summary sessions: 81006
Summary transactions: 1499
Summary revenue: 104458.0


In [12]:
revenue_matrix = (
    pre_df.pivot(
        index="week_start_date",
        columns="geo_region",
        values="observed_revenue_usd"
    )
    .sort_index()
)

session_matrix = (
    pre_df.pivot(
        index="week_start_date",
        columns="geo_region",
        values="observed_session_count"
    )
    .sort_index()
)

transaction_matrix = (
    pre_df.pivot(
        index="week_start_date",
        columns="geo_region",
        values="observed_transaction_count"
    )
    .sort_index()
)

assert revenue_matrix.shape == (8, 20)
assert session_matrix.shape == (8, 20)
assert transaction_matrix.shape == (8, 20)

assert not revenue_matrix.isna().any().any()
assert not session_matrix.isna().any().any()
assert not transaction_matrix.isna().any().any()

print("Revenue matrix:", revenue_matrix.shape)
print("Session matrix:", session_matrix.shape)
print("Transaction matrix:", transaction_matrix.shape)
print("MATCHING MATRICES: PASS")

display(revenue_matrix)

Revenue matrix: (8, 20)
Session matrix: (8, 20)
Transaction matrix: (8, 20)
MATCHING MATRICES: PASS


geo_region,Arizona,California,Colorado,Florida,Georgia,Illinois,Maryland,Massachusetts,Michigan,New Jersey,New York,North Carolina,Ohio,Oregon,Pennsylvania,Tennessee,Texas,Utah,Virginia,Washington
week_start_date,,,,,,,,,,,,,,,,,,,,
2020-11-01,137.0,1782.0,130.0,472.0,636.0,214.0,320.0,1154.0,353.0,240.0,387.0,50.0,35.0,219.0,188.0,134.0,702.0,23.0,121.0,211.0
2020-11-08,22.0,3094.0,38.0,480.0,226.0,485.0,68.0,212.0,358.0,517.0,561.0,409.0,182.0,146.0,22.0,92.0,714.0,82.0,577.0,538.0
2020-11-15,294.0,3558.0,209.0,543.0,502.0,377.0,0.0,660.0,139.0,793.0,407.0,517.0,59.0,50.0,600.0,198.0,1310.0,119.0,551.0,48.0
2020-11-22,141.0,2922.0,654.0,678.0,72.0,661.0,110.0,841.0,273.0,1684.0,1289.0,405.0,464.0,305.0,236.0,279.0,2572.0,218.0,659.0,744.0
2020-11-29,765.0,4927.0,139.0,770.0,430.0,109.0,177.0,387.0,606.0,770.0,707.0,432.0,474.0,321.0,202.0,180.0,2211.0,167.0,1469.0,605.0
2020-12-06,440.0,6211.0,791.0,1549.0,429.0,828.0,245.0,592.0,362.0,386.0,951.0,1509.0,211.0,396.0,727.0,998.0,2125.0,80.0,1587.0,872.0
2020-12-13,1552.0,3687.0,249.0,2220.0,374.0,1022.0,306.0,707.0,1224.0,763.0,1089.0,379.0,413.0,351.0,778.0,112.0,1488.0,458.0,2505.0,493.0
2020-12-20,169.0,1093.0,0.0,372.0,197.0,61.0,0.0,189.0,363.0,71.0,77.0,204.0,0.0,264.0,471.0,41.0,558.0,57.0,404.0,88.0


## Matching Feature Engineering

The experiment design uses PRE-period information only.

Each eligible region is represented by features describing:

- traffic and revenue scale;
- transaction efficiency;
- weekly revenue stability;
- baseline traffic and revenue direction.

Raw scale measures are log-transformed to reduce the influence of exceptionally
large markets. The selected features are subsequently standardized so that
variables measured in dollars, percentages, and counts contribute on comparable
scales.

A region's TEST-period outcomes are not used in feature construction.

In [13]:
from sklearn.preprocessing import StandardScaler


def percentage_slope(values):
    """
    Estimate the linear weekly trend relative to the series mean.

    A result of 5.0 means that the fitted weekly increase equals
    approximately 5% of the region's average weekly level.
    """

    values = np.asarray(values, dtype=float)
    week_numbers = np.arange(len(values), dtype=float)

    series_mean = values.mean()

    if np.isclose(series_mean, 0):
        return np.nan

    fitted_slope = np.polyfit(
        week_numbers,
        values,
        deg=1
    )[0]

    return 100 * fitted_slope / series_mean

In [14]:
feature_rows = []

for region, region_data in pre_df.groupby("geo_region"):

    region_data = (
        region_data
        .sort_values("week_start_date")
        .reset_index(drop=True)
    )

    sessions = region_data["observed_session_count"].to_numpy(dtype=float)
    transactions = region_data["observed_transaction_count"].to_numpy(dtype=float)
    revenue = region_data["observed_revenue_usd"].to_numpy(dtype=float)

    total_sessions = sessions.sum()
    total_transactions = transactions.sum()
    total_revenue = revenue.sum()

    average_weekly_revenue = revenue.mean()
    weekly_revenue_stddev = revenue.std(ddof=1)

    revenue_cv = (
        weekly_revenue_stddev / average_weekly_revenue
        if average_weekly_revenue > 0
        else np.nan
    )

    feature_rows.append(
        {
            "geo_region": region,
            "pre_weeks": region_data["week_start_date"].nunique(),

            # Scale measures
            "pre_sessions": total_sessions,
            "pre_transactions": total_transactions,
            "pre_revenue_usd": total_revenue,

            # Efficiency diagnostics
            "pre_transaction_rate_pct":
                100 * total_transactions / total_sessions,

            "pre_revenue_per_session_usd":
                total_revenue / total_sessions,

            "pre_average_order_value_usd":
                total_revenue / total_transactions,

            # Stability diagnostics
            "average_weekly_revenue_usd": average_weekly_revenue,
            "weekly_revenue_stddev_usd": weekly_revenue_stddev,
            "revenue_cv": revenue_cv,
            "zero_revenue_weeks": int((revenue == 0).sum()),

            # Baseline directions
            "session_trend_pct_per_week":
                percentage_slope(sessions),

            "transaction_trend_pct_per_week":
                percentage_slope(transactions),

            "revenue_trend_pct_per_week":
                percentage_slope(revenue),

            # Existing warehouse value used for reconciliation
            "warehouse_revenue_cv":
                region_data[
                    "pre_revenue_coefficient_of_variation"
                ].iloc[0]
        }
    )

region_features = (
    pd.DataFrame(feature_rows)
    .sort_values("geo_region")
    .reset_index(drop=True)
)

display(region_features)

,geo_region,pre_weeks,pre_sessions,pre_transactions,pre_revenue_usd,pre_transaction_rate_pct,pre_revenue_per_session_usd,pre_average_order_value_usd,average_weekly_revenue_usd,weekly_revenue_stddev_usd,revenue_cv,zero_revenue_weeks,session_trend_pct_per_week,transaction_trend_pct_per_week,revenue_trend_pct_per_week,warehouse_revenue_cv
0,Arizona,8,1403.0,42.0,3520.0,2.993585,2.508909,83.809524,440.000,506.085820,1.150195,0,1.744561,16.326531,24.177489,1.150195
1,California,8,20964.0,383.0,27274.0,1.826941,1.300992,71.211488,3409.250,1630.179285,0.478164,0,3.422647,7.236106,2.830535,0.478164
2,Colorado,8,1672.0,26.0,2210.0,1.555024,1.321770,85.000000,276.250,289.391356,1.047571,1,3.804967,8.058608,5.929757,1.047571
3,Florida,8,4575.0,86.0,7084.0,1.879781,1.548415,82.372093,885.500,653.989952,0.738554,0,2.358574,-1.550388,14.936409,0.738554
4,Georgia,8,2749.0,38.0,2866.0,1.382321,1.042561,75.421053,358.250,182.694710,0.509964,0,3.447141,1.503759,-7.290732,0.509964
5,Illinois,8,3757.0,62.0,3757.0,1.650253,1.000000,60.596774,469.625,346.690450,0.738228,0,3.221922,4.608295,6.121906,0.738228
6,Maryland,8,1442.0,20.0,1226.0,1.386963,0.850208,61.300000,153.250,128.857341,0.840831,2,2.470114,7.619048,-1.926513,0.840831
7,Massachusetts,8,3929.0,68.0,4742.0,1.730720,1.206923,69.735294,592.750,326.129575,0.550198,0,2.038565,-2.521008,-9.917455,0.550198
8,Michigan,8,2464.0,46.0,3678.0,1.866883,1.492695,79.956522,459.750,334.639998,0.727874,0,3.687384,10.766046,13.987933,0.727874
9,New Jersey,8,2948.0,46.0,5224.0,1.560380,1.772049,113.565217,653.000,493.880842,0.756326,0,2.112813,2.070393,-3.806607,0.756326


In [15]:
assert len(region_features) == 20, \
    "Expected one feature row for each of the 20 eligible regions."

assert region_features["geo_region"].is_unique, \
    "Each region must appear exactly once."

assert region_features["pre_weeks"].eq(8).all(), \
    "Each region must contain eight PRE weeks."

assert int(region_features["pre_sessions"].sum()) == 81006, \
    "Regional sessions do not reconcile."

assert int(region_features["pre_transactions"].sum()) == 1499, \
    "Regional transactions do not reconcile."

assert np.isclose(
    region_features["pre_revenue_usd"].sum(),
    104458.00,
    atol=0.01
), "Regional revenue does not reconcile."

assert np.allclose(
    region_features["revenue_cv"],
    region_features["warehouse_revenue_cv"],
    rtol=1e-6,
    atol=1e-6
), "Notebook and warehouse revenue CV values do not reconcile."

numeric_feature_data = region_features.select_dtypes(include=np.number)

assert not numeric_feature_data.isna().any().any(), \
    "Feature table contains missing numeric values."

assert np.isfinite(numeric_feature_data).all().all(), \
    "Feature table contains infinite values."

print("Feature rows:", len(region_features))
print("Regions:", region_features["geo_region"].nunique())
print("Missing numeric values:", numeric_feature_data.isna().sum().sum())
print("Infinite numeric values:", np.isinf(numeric_feature_data).sum().sum())
print("FEATURE ENGINEERING: PASS")

Feature rows: 20
Regions: 20
Missing numeric values: 0
Infinite numeric values: 0
FEATURE ENGINEERING: PASS


In [16]:
feature_range_summary = (
    region_features[
        [
            "pre_sessions",
            "pre_transactions",
            "pre_revenue_usd",
            "pre_transaction_rate_pct",
            "pre_revenue_per_session_usd",
            "pre_average_order_value_usd",
            "revenue_cv",
            "session_trend_pct_per_week",
            "transaction_trend_pct_per_week",
            "revenue_trend_pct_per_week"
        ]
    ]
    .agg(["min", "median", "mean", "max"])
    .T
    .round(4)
)

display(feature_range_summary)

,min,median,mean,max
pre_sessions,945.0000,2828.0000,4050.3000,20964.0000
pre_transactions,20.0000,46.0000,74.9500,383.0000
pre_revenue_usd,1204.0000,3638.5000,5222.9000,27274.0000
pre_transaction_rate_pct,1.2888,1.8443,1.9338,2.9936
pre_revenue_per_session_usd,0.7402,1.2875,1.3351,2.5089
pre_average_order_value_usd,44.5926,68.5129,69.7654,113.5652
revenue_cv,0.4450,0.7384,0.7590,1.2159
session_trend_pct_per_week,0.9477,3.3223,3.0488,4.2885
transaction_trend_pct_per_week,-2.5210,7.8388,7.5775,18.9662
revenue_trend_pct_per_week,-9.9175,6.6259,7.1655,24.1775


In [17]:
region_features["log_pre_sessions"] = np.log1p(
    region_features["pre_sessions"]
)

region_features["log_pre_revenue"] = np.log1p(
    region_features["pre_revenue_usd"]
)

matching_feature_columns = [
    "log_pre_sessions",
    "log_pre_revenue",
    "pre_transaction_rate_pct",
    "revenue_cv",
    "session_trend_pct_per_week",
    "revenue_trend_pct_per_week"
]

print("Selected matching features:")

for feature in matching_feature_columns:
    print("-", feature)

Selected matching features:
- log_pre_sessions
- log_pre_revenue
- pre_transaction_rate_pct
- revenue_cv
- session_trend_pct_per_week
- revenue_trend_pct_per_week


In [18]:
correlation_columns = [
    "log_pre_sessions",
    "log_pre_revenue",
    "pre_transaction_rate_pct",
    "pre_revenue_per_session_usd",
    "pre_average_order_value_usd",
    "revenue_cv",
    "session_trend_pct_per_week",
    "transaction_trend_pct_per_week",
    "revenue_trend_pct_per_week"
]

feature_correlations = (
    region_features[correlation_columns]
    .corr()
    .round(2)
)

display(feature_correlations)

,log_pre_sessions,log_pre_revenue,pre_transaction_rate_pct,pre_revenue_per_session_usd,pre_average_order_value_usd,revenue_cv,session_trend_pct_per_week,transaction_trend_pct_per_week,revenue_trend_pct_per_week
log_pre_sessions,1.00,0.92,-0.37,-0.22,0.07,-0.55,0.18,-0.23,-0.22
log_pre_revenue,0.92,1.00,-0.11,0.17,0.30,-0.47,0.12,-0.11,-0.06
pre_transaction_rate_pct,-0.37,-0.11,1.00,0.68,-0.19,0.15,-0.09,0.53,0.56
pre_revenue_per_session_usd,-0.22,0.17,0.68,1.00,0.57,0.25,-0.22,0.33,0.42
pre_average_order_value_usd,0.07,0.30,-0.19,0.57,1.00,0.13,-0.15,-0.19,-0.13
revenue_cv,-0.55,-0.47,0.15,0.25,0.13,1.00,0.07,0.36,0.44
session_trend_pct_per_week,0.18,0.12,-0.09,-0.22,-0.15,0.07,1.00,0.24,-0.01
transaction_trend_pct_per_week,-0.23,-0.11,0.53,0.33,-0.19,0.36,0.24,1.00,0.70
revenue_trend_pct_per_week,-0.22,-0.06,0.56,0.42,-0.13,0.44,-0.01,0.70,1.00


In [19]:
scaler = StandardScaler()

scaled_array = scaler.fit_transform(
    region_features[matching_feature_columns]
)

standardized_feature_columns = [
    f"z_{column}"
    for column in matching_feature_columns
]

standardized_features = pd.DataFrame(
    scaled_array,
    columns=standardized_feature_columns
)

matching_profiles = pd.concat(
    [
        region_features[["geo_region"]].reset_index(drop=True),
        standardized_features
    ],
    axis=1
)

display(matching_profiles)

,geo_region,z_log_pre_sessions,z_log_pre_revenue,z_pre_transaction_rate_pct,z_revenue_cv,z_session_trend_pct_per_week,z_revenue_trend_pct_per_week
0,Arizona,-1.032031,-0.102517,2.160048,1.859392,-1.395383,1.957139
1,California,2.704140,2.702849,-0.217743,-1.335189,0.399940,-0.498718
2,Colorado,-0.789783,-0.740129,-0.771951,1.371554,0.808970,-0.142169
3,Florida,0.600755,0.855663,-0.110048,-0.097391,-0.738473,0.894000
4,Georgia,-0.102973,-0.384089,-1.123945,-1.184021,0.426145,-1.663118
5,Illinois,0.328594,-0.013251,-0.577860,-0.098941,0.185192,-0.120063
6,Maryland,-0.994167,-1.547076,-1.114485,0.388793,-0.619140,-1.045992
7,Massachusetts,0.390440,0.305737,-0.413856,-0.992768,-1.080838,-1.965308
8,Michigan,-0.254172,-0.042365,-0.136336,-0.148162,0.683172,0.784883
9,New Jersey,-0.006422,0.438364,-0.761035,-0.012912,-1.001404,-1.262287


In [20]:
standardized_values = matching_profiles[
    standardized_feature_columns
]

standardized_means = standardized_values.mean()
standardized_stddevs = standardized_values.std(ddof=0)

assert matching_profiles.shape == (20, 7), \
    "Expected one region column plus six standardized features."

assert not standardized_values.isna().any().any(), \
    "Standardized features contain missing values."

assert np.isfinite(standardized_values).all().all(), \
    "Standardized features contain infinite values."

assert np.allclose(
    standardized_means,
    0,
    atol=1e-10
), "Standardized feature means are not zero."

assert np.allclose(
    standardized_stddevs,
    1,
    atol=1e-10
), "Standardized feature standard deviations are not one."

print("Matching-profile shape:", matching_profiles.shape)
print(
    "Maximum absolute standardized mean:",
    standardized_means.abs().max()
)
print(
    "Maximum deviation from unit standard deviation:",
    (standardized_stddevs - 1).abs().max()
)
print("STANDARDIZATION: PASS")

Matching-profile shape: (20, 7)
Maximum absolute standardized mean: 1.3322676295501878e-15
Maximum deviation from unit standard deviation: 2.220446049250313e-16
STANDARDIZATION: PASS


In [21]:
region_features.to_csv(
    "geo_region_matching_features.csv",
    index=False
)

matching_profiles.to_csv(
    "geo_region_matching_features_standardized.csv",
    index=False
)

print("Feature CSV files created.")

Feature CSV files created.


## Pairwise Geo Similarity

Every possible unordered pair of eligible regions is evaluated using PRE-period
information only.

### Primary summary-feature distance

Because log sessions and log revenue are highly correlated, they are combined
into one standardized scale index. The primary feature distance therefore gives
equal conceptual representation to:

1. market scale;
2. transaction efficiency;
3. revenue volatility;
4. session direction;
5. revenue direction.

### Sensitivity distance

A second Euclidean distance retains all six original standardized variables.
This tests whether combining the two scale variables materially changes the
ranking of candidate pairs.

### Weekly trajectory diagnostics

Pair similarity is also evaluated using:

- session correlation;
- revenue correlation;
- mean-normalized session RMSE;
- mean-normalized revenue RMSE;
- session, transaction, and revenue scale ratios.

No treatment or control assignments are made at this stage.

In [22]:
matching_profiles["scale_index"] = (
    matching_profiles[
        [
            "z_log_pre_sessions",
            "z_log_pre_revenue"
        ]
    ]
    .mean(axis=1)
)

scale_index_scaler = StandardScaler()

matching_profiles["z_scale_index"] = (
    scale_index_scaler
    .fit_transform(
        matching_profiles[["scale_index"]]
    )
    .ravel()
)

primary_feature_columns = [
    "z_scale_index",
    "z_pre_transaction_rate_pct",
    "z_revenue_cv",
    "z_session_trend_pct_per_week",
    "z_revenue_trend_pct_per_week"
]

sensitivity_feature_columns = [
    "z_log_pre_sessions",
    "z_log_pre_revenue",
    "z_pre_transaction_rate_pct",
    "z_revenue_cv",
    "z_session_trend_pct_per_week",
    "z_revenue_trend_pct_per_week"
]

print("Primary matching dimensions:", len(primary_feature_columns))
print("Sensitivity dimensions:", len(sensitivity_feature_columns))
print(
    "Scale-index mean:",
    matching_profiles["z_scale_index"].mean()
)
print(
    "Scale-index population standard deviation:",
    matching_profiles["z_scale_index"].std(ddof=0)
)

Primary matching dimensions: 5
Sensitivity dimensions: 6
Scale-index mean: -1.734723475976807e-17
Scale-index population standard deviation: 0.9999999999999999


In [24]:
from itertools import combinations


def safe_correlation(left_values, right_values):
    """
    Return Pearson correlation unless either series is constant.
    """

    left_values = np.asarray(left_values, dtype=float)
    right_values = np.asarray(right_values, dtype=float)

    if np.isclose(left_values.std(ddof=0), 0):
        return np.nan

    if np.isclose(right_values.std(ddof=0), 0):
        return np.nan

    return np.corrcoef(left_values, right_values)[0, 1]


def mean_normalized_shape_rmse(left_values, right_values):
    """
    Compare weekly trajectory shapes after dividing each series
    by its own mean.

    Lower values indicate more similar relative weekly patterns.
    """

    left_values = np.asarray(left_values, dtype=float)
    right_values = np.asarray(right_values, dtype=float)

    left_mean = left_values.mean()
    right_mean = right_values.mean()

    if np.isclose(left_mean, 0) or np.isclose(right_mean, 0):
        return np.nan

    normalized_left = left_values / left_mean
    normalized_right = right_values / right_mean

    return np.sqrt(
        np.mean(
            (normalized_left - normalized_right) ** 2
        )
    )


def scale_ratio(left_value, right_value):
    """
    Express relative scale from 0 to 1.

    A ratio of 1 means identical scale.
    A ratio of 0.50 means the smaller market is half the size
    of the larger market.
    """

    left_value = float(left_value)
    right_value = float(right_value)

    larger_value = max(left_value, right_value)

    if np.isclose(larger_value, 0):
        return np.nan

    return min(left_value, right_value) / larger_value

In [25]:
profile_lookup = (
    matching_profiles
    .set_index("geo_region")
)

regional_totals_lookup = (
    region_features
    .set_index("geo_region")
)

eligible_regions = sorted(
    profile_lookup.index.tolist()
)

assert len(eligible_regions) == 20

print("Eligible regions:", len(eligible_regions))
print("Expected unordered pairs:", len(list(combinations(eligible_regions, 2))))

Eligible regions: 20
Expected unordered pairs: 190


In [26]:
pair_rows = []

for region_a, region_b in combinations(eligible_regions, 2):

    primary_a = (
        profile_lookup
        .loc[region_a, primary_feature_columns]
        .to_numpy(dtype=float)
    )

    primary_b = (
        profile_lookup
        .loc[region_b, primary_feature_columns]
        .to_numpy(dtype=float)
    )

    sensitivity_a = (
        profile_lookup
        .loc[region_a, sensitivity_feature_columns]
        .to_numpy(dtype=float)
    )

    sensitivity_b = (
        profile_lookup
        .loc[region_b, sensitivity_feature_columns]
        .to_numpy(dtype=float)
    )

    sessions_a = (
        session_matrix[region_a]
        .sort_index()
        .to_numpy(dtype=float)
    )

    sessions_b = (
        session_matrix[region_b]
        .sort_index()
        .to_numpy(dtype=float)
    )

    revenue_a = (
        revenue_matrix[region_a]
        .sort_index()
        .to_numpy(dtype=float)
    )

    revenue_b = (
        revenue_matrix[region_b]
        .sort_index()
        .to_numpy(dtype=float)
    )

    totals_a = regional_totals_lookup.loc[region_a]
    totals_b = regional_totals_lookup.loc[region_b]

    pair_rows.append(
        {
            "pair_id": f"{region_a}__{region_b}",
            "region_a": region_a,
            "region_b": region_b,

            "category_balanced_feature_distance":
                np.linalg.norm(primary_a - primary_b),

            "six_feature_distance_sensitivity":
                np.linalg.norm(sensitivity_a - sensitivity_b),

            "session_correlation":
                safe_correlation(sessions_a, sessions_b),

            "revenue_correlation":
                safe_correlation(revenue_a, revenue_b),

            "session_shape_nrmse":
                mean_normalized_shape_rmse(
                    sessions_a,
                    sessions_b
                ),

            "revenue_shape_nrmse":
                mean_normalized_shape_rmse(
                    revenue_a,
                    revenue_b
                ),

            "session_scale_ratio":
                scale_ratio(
                    totals_a["pre_sessions"],
                    totals_b["pre_sessions"]
                ),

            "transaction_scale_ratio":
                scale_ratio(
                    totals_a["pre_transactions"],
                    totals_b["pre_transactions"]
                ),

            "revenue_scale_ratio":
                scale_ratio(
                    totals_a["pre_revenue_usd"],
                    totals_b["pre_revenue_usd"]
                )
        }
    )

pair_metrics = (
    pd.DataFrame(pair_rows)
    .sort_values(
        "category_balanced_feature_distance"
    )
    .reset_index(drop=True)
)

display(pair_metrics.head(20))

,pair_id,region_a,region_b,category_balanced_feature_distance,six_feature_distance_sensitivity,session_correlation,revenue_correlation,session_shape_nrmse,revenue_shape_nrmse,session_scale_ratio,transaction_scale_ratio,revenue_scale_ratio
0,Michigan__North Carolina,Michigan,North Carolina,0.955603,0.958124,0.905130,-0.100181,0.086902,1.132776,0.997167,1.000000,0.941869
1,Illinois__North Carolina,Illinois,North Carolina,1.140562,1.251722,0.961677,0.502292,0.057494,0.774222,0.657706,0.741935,0.962100
2,Illinois__Michigan,Illinois,Michigan,1.166901,1.266784,0.931023,0.484670,0.067770,0.696189,0.655842,0.741935,0.978973
3,Colorado__North Carolina,Colorado,North Carolina,1.237968,1.406774,0.948494,0.748353,0.070104,0.658620,0.676649,0.565217,0.565941
4,Illinois__Ohio,Illinois,Ohio,1.246317,1.487836,0.826058,0.440096,0.109986,0.794432,0.660900,0.516129,0.489220
5,Massachusetts__New Jersey,Massachusetts,New Jersey,1.264658,1.325251,0.903948,0.274083,0.072228,0.752203,0.750318,0.676471,0.907734
6,Illinois__New York,Illinois,New York,1.350049,1.510083,0.971438,0.787353,0.043541,0.425869,0.559327,0.574074,0.687089
7,Maryland__Ohio,Maryland,Ohio,1.374688,1.522372,0.883298,0.309633,0.096560,0.932495,0.580749,0.625000,0.667029
8,Colorado__Ohio,Colorado,Ohio,1.384516,1.502173,0.902820,0.389481,0.095451,0.994994,0.673379,0.812500,0.831674
9,New York__Texas,New York,Texas,1.446805,1.683288,0.953132,0.810976,0.056729,0.327986,0.906233,0.654545,0.468151


In [28]:
pair_numeric_columns = [
    "category_balanced_feature_distance",
    "six_feature_distance_sensitivity",
    "session_correlation",
    "revenue_correlation",
    "session_shape_nrmse",
    "revenue_shape_nrmse",
    "session_scale_ratio",
    "transaction_scale_ratio",
    "revenue_scale_ratio"
]

assert pair_metrics.shape == (190, 12), \
    "Expected 190 pairs and 12 columns."

assert pair_metrics["pair_id"].is_unique, \
    "Pair identifiers must be unique."

assert not pair_metrics[
    pair_numeric_columns
].isna().any().any(), \
    "One or more pair metrics are undefined."

assert np.isfinite(
    pair_metrics[pair_numeric_columns]
).all().all(), \
    "One or more pair metrics are infinite."

assert (
    pair_metrics[
        [
            "category_balanced_feature_distance",
            "six_feature_distance_sensitivity",
            "session_shape_nrmse",
            "revenue_shape_nrmse"
        ]
    ] >= 0
).all().all(), "Distance metrics cannot be negative."

assert (
    pair_metrics[
        [
            "session_correlation",
            "revenue_correlation"
        ]
    ].abs() <= 1 + 1e-12
).all().all(), "Correlations must remain between -1 and 1."

assert (
    pair_metrics[
        [
            "session_scale_ratio",
            "transaction_scale_ratio",
            "revenue_scale_ratio"
        ]
    ] > 0
).all().all(), "Scale ratios must be greater than zero."

assert (
    pair_metrics[
        [
            "session_scale_ratio",
            "transaction_scale_ratio",
            "revenue_scale_ratio"
        ]
    ] <= 1
).all().all(), "Scale ratios cannot exceed one."

print("Pair rows:", len(pair_metrics))
print("Unique pair IDs:", pair_metrics["pair_id"].nunique())
print(
    "Undefined correlations:",
    pair_metrics[
        ["session_correlation", "revenue_correlation"]
    ].isna().sum().sum()
)
print("PAIRWISE SIMILARITY: PASS")

Pair rows: 190
Unique pair IDs: 190
Undefined correlations: 0
PAIRWISE SIMILARITY: PASS


In [29]:
pair_metric_summary = (
    pair_metrics[pair_numeric_columns]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .T
    .round(4)
)

display(pair_metric_summary)

,count,mean,std,min,10%,25%,50%,75%,90%,max
category_balanced_feature_distance,190.0,3.0701,1.0520,0.9556,1.7231,2.2737,3.0428,3.7947,4.3746,6.0482
six_feature_distance_sensitivity,190.0,3.3550,1.1759,0.9581,1.9387,2.4664,3.2764,4.1374,4.7837,6.9565
session_correlation,190.0,0.8758,0.0852,0.5372,0.7455,0.8326,0.8997,0.9412,0.9629,0.9935
revenue_correlation,190.0,0.3767,0.3041,-0.4538,-0.0404,0.1484,0.4138,0.6043,0.7661,0.9507
session_shape_nrmse,190.0,0.0934,0.0311,0.0348,0.0554,0.0690,0.0904,0.1141,0.1335,0.1881
revenue_shape_nrmse,190.0,0.7984,0.2448,0.2717,0.4815,0.6401,0.7737,0.9463,1.1065,1.5794
session_scale_ratio,190.0,0.5122,0.2535,0.0451,0.1790,0.3269,0.4869,0.6805,0.8773,0.9972
transaction_scale_ratio,190.0,0.5329,0.2479,0.0522,0.1844,0.3300,0.5349,0.7364,0.8438,1.0000
revenue_scale_ratio,190.0,0.5125,0.2530,0.0441,0.1728,0.3142,0.5200,0.6885,0.8962,0.9912


In [30]:
top_feature_pairs = (
    pair_metrics
    .nsmallest(
        20,
        "category_balanced_feature_distance"
    )
    [
        [
            "region_a",
            "region_b",
            "category_balanced_feature_distance",
            "six_feature_distance_sensitivity",
            "session_correlation",
            "revenue_correlation",
            "session_shape_nrmse",
            "revenue_shape_nrmse",
            "session_scale_ratio",
            "revenue_scale_ratio"
        ]
    ]
    .round(4)
)

display(top_feature_pairs)

,region_a,region_b,category_balanced_feature_distance,six_feature_distance_sensitivity,session_correlation,revenue_correlation,session_shape_nrmse,revenue_shape_nrmse,session_scale_ratio,revenue_scale_ratio
0,Michigan,North Carolina,0.9556,0.9581,0.9051,-0.1002,0.0869,1.1328,0.9972,0.9419
1,Illinois,North Carolina,1.1406,1.2517,0.9617,0.5023,0.0575,0.7742,0.6577,0.9621
2,Illinois,Michigan,1.1669,1.2668,0.9310,0.4847,0.0678,0.6962,0.6558,0.9790
3,Colorado,North Carolina,1.2380,1.4068,0.9485,0.7484,0.0701,0.6586,0.6766,0.5659
4,Illinois,Ohio,1.2463,1.4878,0.8261,0.4401,0.1100,0.7944,0.6609,0.4892
5,Massachusetts,New Jersey,1.2647,1.3253,0.9039,0.2741,0.0722,0.7522,0.7503,0.9077
6,Illinois,New York,1.3500,1.5101,0.9714,0.7874,0.0435,0.4259,0.5593,0.6871
7,Maryland,Ohio,1.3747,1.5224,0.8833,0.3096,0.0966,0.9325,0.5807,0.6670
8,Colorado,Ohio,1.3845,1.5022,0.9028,0.3895,0.0955,0.9950,0.6734,0.8317
9,New York,Texas,1.4468,1.6833,0.9531,0.8110,0.0567,0.3280,0.9062,0.4682


In [31]:
pair_metrics.to_csv(
    "geo_pair_similarity_metrics.csv",
    index=False
)

pair_metric_summary.to_csv(
    "geo_pair_metric_distribution.csv"
)

print("Pairwise output files created.")

Pairwise output files created.
